# 🛒 Retail Shop Sales Prediction Model
### A Step-by-Step Guide for Beginners on Google Colab

---

**Welcome!** In this notebook, you will learn how to:
1. Load and explore a retail shop dataset
2. Clean and prepare the data
3. Visualize patterns in the data
4. Build a machine learning model to **predict Total Sales**
5. Evaluate how good your model is
6. Make new predictions

> 💡 **Tip:** Run each cell one by one by clicking the ▶ button or pressing `Shift + Enter`.

---
## 📌 Step 0: Upload the Dataset

First, you need to upload the `retail_shop_dataset.csv` file to Google Colab.

**Option A – Upload manually:**
1. Click the 📁 folder icon in the left sidebar
2. Click the upload icon (⬆)
3. Select `retail_shop_dataset.csv` from your computer

**Option B – Run the cell below to upload via code:**

In [ ]:
# Option B: Run this cell to upload the file
from google.colab import files
uploaded = files.upload()  # A dialog will appear — select retail_shop_dataset.csv
print("File uploaded successfully!")

---
## 📌 Step 1: Install & Import Libraries

Python has many helpful libraries. We'll use:
- **pandas** – to work with data (like Excel in Python)
- **numpy** – for numbers and math
- **matplotlib / seaborn** – for charts and graphs
- **scikit-learn** – for building machine learning models

In [ ]:
# All these libraries come pre-installed in Google Colab — no installation needed!

import pandas as pd          # Data manipulation
import numpy as np           # Numerical operations
import matplotlib.pyplot as plt  # Plotting
import seaborn as sns        # Advanced plotting

# Machine Learning tools from scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import warnings
warnings.filterwarnings('ignore')

# Make plots look nice
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

print("✅ All libraries imported successfully!")

---
## 📌 Step 2: Load the Dataset

Now let's load our retail shop data into a **DataFrame** (think of it as a table).

In [ ]:
# Load the CSV file into a DataFrame
df = pd.read_csv('retail_shop_dataset.csv')

# Display the first 5 rows
print(f"✅ Dataset loaded! Shape: {df.shape[0]} rows × {df.shape[1]} columns\n")
df.head()

### 📋 What does each column mean?

| Column | Description |
|---|---|
| `Transaction_ID` | Unique ID for each sale |
| `Date` | Date of the transaction |
| `Day_of_Week` | Day name (Monday, Tuesday, …) |
| `Month` | Month number (1=January, 12=December) |
| `Is_Weekend` | 1 if weekend, 0 if weekday |
| `Store` | Which store (A, B, C, D) |
| `Category` | Product category (Electronics, Clothing, …) |
| `Unit_Price` | Price of one item |
| `Units_Sold` | Number of items sold |
| `Discount_Percent` | Discount given (%) |
| `Discount_Amount` | Total discount amount |
| `Total_Sales` | **Total revenue (our target to predict!)** |
| `Customer_Age` | Age of the customer |
| `Customer_Gender` | Male or Female |
| `Payment_Method` | Cash, Credit Card, Debit Card, UPI |
| `Customer_Rating` | Rating given by customer (1.0 – 5.0) |

---
## 📌 Step 3: Explore the Data (EDA)

Before building a model, we need to **understand** our data. This is called **Exploratory Data Analysis (EDA)**.

In [ ]:
# Check the data types and non-null counts
print("=== Dataset Info ===")
df.info()

In [ ]:
# Get basic statistics (count, mean, min, max, etc.)
print("=== Basic Statistics ===")
df.describe()

In [ ]:
# Check for missing values
print("=== Missing Values ===")
missing = df.isnull().sum()
print(missing)
print(f"\n✅ Total missing values: {missing.sum()}")

In [ ]:
# Check unique values in categorical columns
print("Unique Stores:", df['Store'].unique())
print("Unique Categories:", df['Category'].unique())
print("Unique Payment Methods:", df['Payment_Method'].unique())

### 📊 Visualize the Data

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Retail Shop Data - Overview', fontsize=16, fontweight='bold')

# 1. Total Sales distribution
axes[0, 0].hist(df['Total_Sales'], bins=40, color='steelblue', edgecolor='white')
axes[0, 0].set_title('Distribution of Total Sales')
axes[0, 0].set_xlabel('Total Sales (₹)')
axes[0, 0].set_ylabel('Count')

# 2. Sales by Category
category_sales = df.groupby('Category')['Total_Sales'].mean().sort_values(ascending=False)
axes[0, 1].bar(category_sales.index, category_sales.values, color=sns.color_palette('husl', len(category_sales)))
axes[0, 1].set_title('Average Sales by Category')
axes[0, 1].set_xlabel('Category')
axes[0, 1].set_ylabel('Avg Total Sales (₹)')
axes[0, 1].tick_params(axis='x', rotation=45)

# 3. Sales by Store
store_sales = df.groupby('Store')['Total_Sales'].sum()
axes[1, 0].bar(store_sales.index, store_sales.values, color=sns.color_palette('husl', 4))
axes[1, 0].set_title('Total Sales by Store')
axes[1, 0].set_xlabel('Store')
axes[1, 0].set_ylabel('Total Sales (₹)')

# 4. Sales by Month
month_sales = df.groupby('Month')['Total_Sales'].mean()
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
axes[1, 1].plot(month_sales.index, month_sales.values, marker='o', color='coral', linewidth=2)
axes[1, 1].set_title('Average Sales by Month')
axes[1, 1].set_xlabel('Month')
axes[1, 1].set_ylabel('Avg Total Sales (₹)')
axes[1, 1].set_xticks(range(1, 13))
axes[1, 1].set_xticklabels(month_names, rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap - shows how features relate to each other
plt.figure(figsize=(10, 6))
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols.remove('Transaction_ID')  # Not useful for correlation
corr_matrix = df[numeric_cols].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print("💡 Values close to 1 or -1 = strong relationship with Total_Sales")

In [ ]:
# Scatter plot: Unit Price vs Total Sales
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.scatter(df['Unit_Price'], df['Total_Sales'], alpha=0.4, color='steelblue')
plt.xlabel('Unit Price (₹)')
plt.ylabel('Total Sales (₹)')
plt.title('Unit Price vs Total Sales')

plt.subplot(1, 2, 2)
plt.scatter(df['Units_Sold'], df['Total_Sales'], alpha=0.4, color='coral')
plt.xlabel('Units Sold')
plt.ylabel('Total Sales (₹)')
plt.title('Units Sold vs Total Sales')

plt.tight_layout()
plt.show()

---
## 📌 Step 4: Prepare the Data for Machine Learning

Machine learning models only understand **numbers**. We need to:
1. Drop columns we don't need
2. Convert text columns (like "Male"/"Female") into numbers
3. Split data into **features (X)** and **target (y)**
4. Split into **training set** and **test set**

In [ ]:
# Step 4.1: Drop columns that are not useful for prediction
# Transaction_ID is just a number, Date is already split into Month/Day_of_Week
df_model = df.drop(columns=['Transaction_ID', 'Date'])

print("Columns remaining:", list(df_model.columns))

In [ ]:
# Step 4.2: Convert text (categorical) columns to numbers using Label Encoding
# This converts e.g. 'Male' -> 0, 'Female' -> 1

le = LabelEncoder()
categorical_columns = ['Day_of_Week', 'Store', 'Category', 'Customer_Gender', 'Payment_Method']

for col in categorical_columns:
    df_model[col] = le.fit_transform(df_model[col])
    print(f"  ✅ Encoded '{col}'")

print("\nFirst 3 rows after encoding:")
df_model.head(3)

In [ ]:
# Step 4.3: Define Features (X) and Target (y)
# X = all columns EXCEPT Total_Sales
# y = Total_Sales (what we want to predict)

X = df_model.drop(columns=['Total_Sales'])   # Features
y = df_model['Total_Sales']                  # Target

print(f"Features (X) shape: {X.shape}  →  {X.shape[0]} rows, {X.shape[1]} columns")
print(f"Target  (y) shape: {y.shape}   →  {y.shape[0]} values")
print("\nFeature columns:", list(X.columns))

In [ ]:
# Step 4.4: Split data into Training (80%) and Testing (20%) sets
# Training set → model learns from this
# Testing set  → we check how well the model does on unseen data

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 20% for testing
    random_state=42     # Fixed seed so results are reproducible
)

print(f"Training set size:  {X_train.shape[0]} rows")
print(f"Testing set size:   {X_test.shape[0]} rows")

In [ ]:
# Step 4.5: Scale the features (bring all values to a similar range)
# This helps some models learn better

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # Fit on training data only!
X_test_scaled  = scaler.transform(X_test)        # Apply same scaling to test data

print("✅ Features scaled successfully!")

---
## 📌 Step 5: Build Machine Learning Models

We'll try **two models**:
1. **Linear Regression** – Simple and easy to understand (draws a straight line through data)
2. **Random Forest** – More powerful (combines many decision trees)

We'll compare them and pick the best one!

### 🔵 Model 1: Linear Regression

In [ ]:
# Create and train the Linear Regression model
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)  # Train: model learns from training data

# Make predictions on the test set
lr_predictions = lr_model.predict(X_test_scaled)

print("✅ Linear Regression model trained!")
print(f"\nSample predictions vs actual values:")
comparison = pd.DataFrame({
    'Actual':    y_test.values[:5],
    'Predicted': lr_predictions[:5].round(2)
})
print(comparison.to_string(index=False))

### 🟢 Model 2: Random Forest Regressor

In [ ]:
# Create and train the Random Forest model
rf_model = RandomForestRegressor(
    n_estimators=100,  # Use 100 decision trees
    random_state=42
)
rf_model.fit(X_train, y_train)  # Random Forest doesn't need scaled features

# Make predictions
rf_predictions = rf_model.predict(X_test)

print("✅ Random Forest model trained!")
print(f"\nSample predictions vs actual values:")
comparison = pd.DataFrame({
    'Actual':    y_test.values[:5],
    'Predicted': rf_predictions[:5].round(2)
})
print(comparison.to_string(index=False))

---
## 📌 Step 6: Evaluate the Models

How do we know if our model is good? We use these metrics:

| Metric | What it means | Lower is better? |
|---|---|---|
| **MAE** (Mean Absolute Error) | Average prediction error (in ₹) | ✅ Yes |
| **RMSE** (Root Mean Squared Error) | Penalizes large errors more | ✅ Yes |
| **R² Score** | How well model explains the data (0 to 1) | ❌ Higher is better |

In [ ]:
def evaluate_model(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    print(f"{'='*40}")
    print(f"  📊 {name}")
    print(f"{'='*40}")
    print(f"  MAE  (Mean Absolute Error) : ₹{mae:,.2f}")
    print(f"  RMSE (Root Mean Sq. Error) : ₹{rmse:,.2f}")
    print(f"  R²   (R-Squared Score)     : {r2:.4f} ({r2*100:.1f}% accuracy)")
    print()
    return {'Model': name, 'MAE': mae, 'RMSE': rmse, 'R2': r2}

lr_results = evaluate_model("Linear Regression", y_test, lr_predictions)
rf_results = evaluate_model("Random Forest",     y_test, rf_predictions)

In [ ]:
# Compare both models side by side
results_df = pd.DataFrame([lr_results, rf_results])
results_df = results_df.set_index('Model')
print("\n📋 Model Comparison:")
print(results_df.to_string())

best_model = results_df['R2'].idxmax()
print(f"\n🏆 Best Model: {best_model} (highest R² score)")

In [ ]:
# Visual comparison: Actual vs Predicted values
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Actual vs Predicted Total Sales', fontsize=14, fontweight='bold')

for ax, name, preds, color in zip(
    axes,
    ['Linear Regression', 'Random Forest'],
    [lr_predictions, rf_predictions],
    ['steelblue', 'coral']
):
    ax.scatter(y_test, preds, alpha=0.4, color=color, s=20)
    # Perfect prediction line
    min_val = min(y_test.min(), preds.min())
    max_val = max(y_test.max(), preds.max())
    ax.plot([min_val, max_val], [min_val, max_val], 'k--', linewidth=1.5, label='Perfect Prediction')
    ax.set_xlabel('Actual Total Sales (₹)')
    ax.set_ylabel('Predicted Total Sales (₹)')
    ax.set_title(name)
    ax.legend()

plt.tight_layout()
plt.show()
print("💡 Points close to the dashed line = accurate predictions!")

In [ ]:
# Feature Importance - which features matter most? (Random Forest only)
feature_importance = pd.Series(
    rf_model.feature_importances_,
    index=X.columns
).sort_values(ascending=True)

plt.figure(figsize=(8, 6))
feature_importance.plot(kind='barh', color='steelblue', edgecolor='white')
plt.title('Feature Importance (Random Forest)', fontsize=13, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()
print("💡 Higher bar = more influence on predicting Total_Sales")

---
## 📌 Step 7: Make New Predictions

Now let's use our trained model to predict sales for a **new transaction**!

In [ ]:
# Let's manually encode the categories (same encoding used during training)
# Re-fit encoders on full dataset to get the mapping
df_model_orig = df.drop(columns=['Transaction_ID', 'Date'])

encoders = {}
for col in categorical_columns:
    enc = LabelEncoder()
    enc.fit(df_model_orig[col])
    encoders[col] = enc

print("Day_of_Week options:",  list(encoders['Day_of_Week'].classes_))
print("Store options:",        list(encoders['Store'].classes_))
print("Category options:",     list(encoders['Category'].classes_))
print("Gender options:",       list(encoders['Customer_Gender'].classes_))
print("Payment options:",      list(encoders['Payment_Method'].classes_))

In [ ]:
# 🛒 Create a new transaction to predict
# Feel free to change these values!

new_transaction = {
    'Day_of_Week':    'Saturday',      # Change to any day
    'Month':          12,              # December (holiday season)
    'Is_Weekend':     1,               # 1=weekend, 0=weekday
    'Store':          'Store_B',       # Change to Store_A/B/C/D
    'Category':       'Electronics',  # Change to any category
    'Unit_Price':     25000,          # Price of item
    'Units_Sold':     3,              # Number of items
    'Discount_Percent': 10,           # Discount %
    'Discount_Amount':  7500,         # = Unit_Price * Units_Sold * Discount_Percent / 100
    'Customer_Age':   35,
    'Customer_Gender': 'Male',
    'Payment_Method': 'Credit Card',
    'Customer_Rating': 4.5
}

# Encode the categorical values
for col in categorical_columns:
    new_transaction[col] = encoders[col].transform([new_transaction[col]])[0]

# Convert to DataFrame with same column order as training
new_df = pd.DataFrame([new_transaction])[X.columns]

# Predict using Random Forest (best model)
predicted_sales = rf_model.predict(new_df)[0]

print("🛒 New Transaction Details:")
print(f"   Store: Store_B | Category: Electronics | Month: December")
print(f"   Unit Price: ₹25,000 | Units Sold: 3 | Discount: 10%")
print(f"\n💰 Predicted Total Sales: ₹{predicted_sales:,.2f}")

---
## 📌 Step 8: Save the Model (Optional)

You can save your trained model so you don't have to train it again next time!

In [ ]:
import joblib

# Save the model
joblib.dump(rf_model, 'retail_sales_model.pkl')
print("✅ Model saved as 'retail_sales_model.pkl'")

# To load it later:
# loaded_model = joblib.load('retail_sales_model.pkl')
# prediction = loaded_model.predict(new_df)

In [ ]:
# Download the saved model to your computer (Google Colab only)
from google.colab import files
files.download('retail_sales_model.pkl')
print("✅ Model downloaded!")

---
## 🎉 Summary

Congratulations! You have just built a complete machine learning pipeline:

| Step | What you did |
|---|---|
| ✅ Step 1 | Imported necessary Python libraries |
| ✅ Step 2 | Loaded the retail shop dataset |
| ✅ Step 3 | Explored and visualized the data (EDA) |
| ✅ Step 4 | Preprocessed the data (encoding, scaling, train/test split) |
| ✅ Step 5 | Trained Linear Regression and Random Forest models |
| ✅ Step 6 | Evaluated and compared model performance |
| ✅ Step 7 | Made predictions on new data |
| ✅ Step 8 | Saved the model for future use |

---
### 🚀 What to Try Next?

1. **Try different models**: `GradientBoostingRegressor`, `XGBoost`, `SVR`
2. **Tune the model**: Use `GridSearchCV` to find the best hyperparameters
3. **Add more features**: Try adding `Unit_Price × Units_Sold` as a new column
4. **Classification task**: Instead of predicting `Total_Sales`, predict `High/Low` sales category
5. **Time series**: Predict future monthly sales using historical trends

---
*Happy Learning! 🐍📊*